# Extended Ablation Study: GS-GLS vs Baselines

This notebook provides an extensive comparison of **Geodesic Spectral-GLS (GS-GLS)** against standard reconciliation baselines across a wide range of hierarchy sizes.

## Comparison Methods
- **OLS**: Identity projection.
- **WLS (Structural)**: Weighted Least Squares with weights inversely proportional to node volume (structural scaling).
- **MinT (Sample)**: Trace minimization with sample covariance.
- **MinT (Shrinkage)**: Trace minimization with diagonal shrinkage (WLS-Variance target).
- **GS-GLS**: The proposed Geodesic Spectral method (Spectral for Stationary, Wavelet for Non-Stationary).

## Scenarios
1. **Stationary**: Homoscedastic noise.
2. **Non-Stationary**: Heteroscedastic noise (variance increases over time).

## Hierarchy Sizes
We test on **20 different hierarchy sizes**, ranging from small (~40 nodes) to XXL (>2000 nodes), to demonstrate scalability and performance stability.

In [ ]:
import numpy as np
import pandas as pd
import time
import random
import sys
import os
import gc
import matplotlib.pyplot as plt
import scipy.sparse as sp

# Add local path if running from subdir
sys.path.append(os.getcwd())

from hierarchy import Hierarchy
from data_generator import HierarchicalDataGenerator
from gs_gls import GSGLS
import baselines

%load_ext autoreload
%autoreload 2

### 1. Configuration: 20 Hierarchy Sizes

In [ ]:
def get_hierarchy_configs():
    """ Returns 20 configs spread from Small (~50) to XXL (~2000+). """
    # Generated to span a smooth curve of node counts
    pairs = [
        (2, 5), (3, 3), (3, 4), (2, 10), (3, 5), (4, 3), (2, 15), 
        (3, 6), (4, 4), (5, 3), (3, 8), (4, 5), (6, 3), (5, 4),
        (3, 12), (4, 6), (5, 5), (6, 4), (4, 8), (5, 6) 
    ]
    
    def approx_nodes(d, b):
        if b == 1: return d+1
        return (b**(d+1) - 1) // (b - 1)
        
    pairs.sort(key=lambda x: approx_nodes(x[0], x[1]))
    
    final_configs = [{'depth': d, 'branch': b, 'id': i} for i, (d, b) in enumerate(pairs)]
    return final_configs[:20]

configs = get_hierarchy_configs()
print(f"Loaded {len(configs)} hierarchy configurations.")

### 2. Helper Method: Structural WLS
Implementing WLS where $W = \text{diag}(v)^{-1}$.

In [ ]:
def wls_struct(y_hat, S):
    """
    Structural Weighted Least Squares.
    Weights are inversely proportional to the volume (number of bottom nodes) of each node.
    """
    if sp.issparse(S):
        volumes = np.array(S.sum(axis=1)).flatten()
    else:
        volumes = np.sum(S, axis=1)
        
    # Weight ~ 1/Volume
    weights = 1.0 / (volumes + 1e-6)
    
    # Construct projection P = S (S' W S)^-1 S' W
    # Here W is the inverse variance matrix (Precision), so W_diagonal = weights.
    if sp.issparse(S):
        W = sp.diags(weights)
        STS = S.T @ W @ S
        solver = sp.linalg.splu(STS)
        RHS = S.T @ (W @ y_hat)
        coeffs = solver.solve(RHS)
        return S @ coeffs
    else:
        W = np.diag(weights)
        STS = S.T @ W @ S
        try:
            STS_inv = np.linalg.inv(STS)
        except:
             STS_inv = np.linalg.pinv(STS)
        return S @ STS_inv @ S.T @ W @ y_hat

### 3. Generation Wrapper

In [ ]:
def generate_random_hierarchy(depth, branching_factor):
    structure = {}
    current_layer = ['Total']
    node_ctr = 1
    for d in range(depth):
        next_layer = []
        for parent in current_layer:
            n_children = random.randint(max(2, branching_factor-2), branching_factor + 2)
            children = []
            for _ in range(n_children):
                child_name = f'Node_{d+1}_{node_ctr}'
                children.append(child_name)
                node_ctr += 1
            structure[parent] = children
            next_layer.extend(children)
        current_layer = next_layer
    return structure

def estimate_memory_gb(n_nodes):
    # Estimate for dense N x N float64 matrix
    return (n_nodes**2 * 8) / (1024**3)

### 4. Main Experiment Loop

In [ ]:
def run_full_ablation(scenarios=['Stationary', 'Non-Stationary']):
    results = []
    
    methods = [
        ('OLS', baselines.ols_identity, False),
        ('WLS (Sample)', wls_struct, False),
        ('MinT (Sample)', baselines.mint_sample, True),
        ('MinT (Shrink)', baselines.mint_shrink, True),
        ('GS-GLS', None, False)
    ]
    
    for sc in scenarios:
        print(f"\n=== Running Scenario: {sc} ===")
        is_hetero = (sc == 'Non-Stationary')
        
        for conf in configs:
            depth, branch = conf['depth'], conf['branch']
            
            # 1. Setup Hierarchy
            h = None
            while h is None:
                try:
                    s = generate_random_hierarchy(depth, branch)
                    h = Hierarchy(s)
                except:
                    pass
            
            n_nodes = h.n_nodes
            print(f"Size {conf['id']}: {n_nodes} nodes (d={depth}, b={branch})...", end='')
            
            # 2. Data
            n_total = 1000
            n_train = 800
            gen = HierarchicalDataGenerator(h, n_timesteps=n_total)
            Y_true = gen.generate_ground_truth()
            E = gen.generate_spatiotemporal_noise(
                spatial_rho=1.5, temporal_ar_coefs=[0.5], noise_scale=1.0, 
                heteroscedastic=is_hetero
            )
            Y_hat = Y_true + E
            
            residuals_train = E[:, :n_train]
            Y_hat_test = Y_hat[:, n_train:]
            Y_true_test = Y_true[:, n_train:]
            
            # Prep MinT
            n_t_block = 10
            S_sp = h.get_summing_matrix()
            # Use sparse S for large graphs if supported, but baselines are mostly dense.
            # S_total construction might be heavy for huge graphs.
            # For very large graphs, we might want to avoid full S_total if not needed.
            # But MinT needs it.
            
            try:
                S_total = baselines.build_spatiotemporal_s(S_sp, n_t_block)
                
                train_blocks = []
                for i in range(0, residuals_train.shape[1] - n_t_block + 1, n_t_block):
                    train_blocks.append(residuals_train[:, i:i+n_t_block].flatten(order='F'))
                residuals_samples = np.array(train_blocks)
            except MemoryError:
                S_total = None # Will fail baselines
                
            
            # 3. Fit GS-GLS
            gs_model_status = None
            gs_time_train = 0
            if 'GS-GLS' in [m[0] for m in methods]:
                try:
                    t0 = time.time()
                    mode = 'wavelet' if is_hetero else 'spectral'
                    gs_inst = GSGLS(h, temporal_method=mode)
                    gs_inst.fit(residuals_train)
                    gs_time_train = time.time() - t0
                    gs_model_status = gs_inst
                except MemoryError:
                    gs_model_status = f"OOM ({estimate_memory_gb(n_nodes):.1f}GB)"
                except Exception as e:
                    gs_model_status = "Error"
            
            # 4. Evaluate
            for name, func, needs_train in methods:
                row = {'Scenario': sc, 'Nodes': n_nodes, 'Method': name, 
                       'MSE': np.nan, 'MAE': np.nan, 'Incoherence': np.nan, 
                       'Train_Time': np.nan, 'Infer_Time': np.nan, 'Total_Time': np.nan}
                
                try:
                    t0_train = time.time()
                    # Baseline Training (if any implicit)
                    if name == 'GS-GLS':
                        if isinstance(gs_model_status, str):
                             raise ValueError(gs_model_status)
                        row['Train_Time'] = gs_time_train
                    else:
                        # Baselines are lazy / no-op for fit usually
                        if S_total is None: raise MemoryError()
                        row['Train_Time'] = 0.0

                    # Inference
                    t0_infer = time.time()
                    mse_sum = 0
                    mae_sum = 0
                    incoh_max = 0
                    count = 0
                    
                    limit = 50 if n_nodes > 1000 else 200 # Limit inference steps for speed
                    
                    for i in range(0, Y_hat_test.shape[1] - n_t_block + 1, n_t_block):
                        if count >= limit: break
                        
                        y_blk = Y_hat_test[:, i:i+n_t_block]
                        y_tru = Y_true_test[:, i:i+n_t_block]
                        
                        if name == 'GS-GLS':
                            y_rec = gs_model_status.reconcile(y_blk)
                        else:
                            y_flat = y_blk.flatten(order='F')
                            if needs_train:
                                y_rec_flat = func(y_flat, residuals_samples, S_total)
                            else:
                                y_rec_flat = func(y_flat, S_total)
                            y_rec = y_rec_flat.reshape((n_nodes, n_t_block), order='F')
                            
                        mse_sum += np.mean((y_rec - y_tru)**2)
                        mae_sum += np.mean(np.abs(y_rec - y_tru))
                        
                        # Incoherence: Check parent - sum(children) for a few nodes
                        # Just checking root
                        diff = np.abs(y_rec[0] - np.sum(y_rec[h.n_nodes-h.m_bottom:], axis=0))
                        if np.max(diff) > incoh_max: incoh_max = np.max(diff)
                        
                        count += 1
                        
                    row['MSE'] = mse_sum / max(1, count)
                    row['MAE'] = mae_sum / max(1, count)
                    row['Incoherence'] = incoh_max
                    row['Infer_Time'] = (time.time() - t0_infer) * (Y_hat_test.shape[1] // n_t_block) / max(1, count)
                    row['Total_Time'] = row['Train_Time'] + row['Infer_Time']
                    
                except MemoryError:
                     row['MSE'] = f"OOM ({estimate_memory_gb(n_nodes):.1f}GB)"
                except Exception as e:
                     if str(e).startswith("OOM"):
                         row['MSE'] = str(e)
                     else:
                         row['MSE'] = "Error"
                
                results.append(row)
            
            print(" Done.")
            gc.collect()
            
    return pd.DataFrame(results)

In [ ]:
df_results = run_full_ablation()
print("\nFinal Experiment Results:")
display(df_results)

### 5. Results Analysis

In [ ]:
# Pivot tables for easier viewing
pivot_mse = df_results.pivot(index='Nodes', columns='Method', values='MSE')
display(pivot_mse)

# Determine OOM threshold
oom_entries = df_results[df_results['MSE'].astype(str).str.contains("OOM")]
if not oom_entries.empty:
    print("OOM encountered at:")
    display(oom_entries[['Nodes', 'Method', 'MSE']])